# Day 2 practice — Building the counter

**Read first:** [03_theory_fastapi_fundamentals.md](03_theory_fastapi_fundamentals.md)

You'll build a real FastAPI app cell by cell, watch the decorator write to the routing table,
trigger the route-order bug on purpose, and read the docs FastAPI generated without being asked.

**Why no `uvicorn` here?** `TestClient` calls the app directly, in-process — same routing, same
validation, same responses, no port to conflict and no server to restart. You'll run the real server
from a terminal at the end.

In [ ]:
from fastapi import APIRouter, FastAPI, HTTPException, Query, status
from fastapi.testclient import TestClient
import json

def show(response, label=""):
    """Print a response the way you'd read it off the wire."""
    print(f"{label:<34} {response.status_code}  {response.text[:150]}")

print("ready")

---

## Part 1 — The smallest app, and what the decorator really did

In [ ]:
app = FastAPI(title="Day 2 demo")

@app.get("/")
def read_root():
    return {"message": "hello"}

client = TestClient(app)
show(client.get("/"), "GET /")

### The decorator is a register — proof

`@app.get("/")` did not modify `read_root`. It wrote a row into the app's routing table and handed
the function back untouched.

In [ ]:
print("read_root is still a plain function:", read_root)
print("calling it directly           :", read_root())
print()
print("the routing table now contains:")
for route in app.routes:
    if hasattr(route, "methods"):
        print(f"   {sorted(route.methods)!s:<22} {route.path:<20} -> {route.endpoint.__name__}")

Those extra rows you didn't write (`/openapi.json`, `/docs`, `/redoc`) are the automatic
documentation. More on those in Part 5.

**The desugared form**, if `@` still feels like magic:

```python
@app.get("/ping")          #  is EXACTLY
def ping(): ...            #

def ping(): ...            #  this
ping = app.get("/ping")(ping)
```

Read the last line right to left: `app.get("/ping")` returns a decorator, that decorator is called
with your function, FastAPI records `(GET, /ping) -> ping`, and gives the function back.

> 🎯 **Import time registers. Request time executes.** Two different clocks.

---

## Part 2 — Path parameters: the type hint is the doorman

In [ ]:
@app.get("/items/{item_id}")
def read_item(item_id: int):
    return {"item_id": item_id, "python_type": type(item_id).__name__}

show(client.get("/items/7"),   "GET /items/7")
show(client.get("/items/abc"), "GET /items/abc")

### 🔮 Predict

`/items/7` returned `"item_id": 7` — an **integer**, not the string `"7"`. URLs are text, so
something converted it.

Before running: what will `/items/7.5` do? And `/items/-3`?

In [ ]:
for path in ["/items/7", "/items/7.5", "/items/-3", "/items/abc", "/items/"]:
    show(client.get(path), f"GET {path}")

`/items/7.5` is a `422` — `7.5` is not an integer, and FastAPI will not silently truncate it.
`/items/-3` works, because nothing said it had to be positive (we'll fix that with `Query`-style
constraints later). `/items/` is a `404` — a different path entirely, with no id at all.

### Reading the 422 properly

In [ ]:
error = client.get("/items/abc").json()
print(json.dumps(error, indent=2))
print()
print("loc reads OUTSIDE IN:", error["detail"][0]["loc"], "-> in the path, the field 'item_id'")
print("input echoes what you sent:", repr(error["detail"][0]["input"]))

---

## Part 3 — 🚨 The route-order bug

This one catches everybody once. Let's make it catch you here, where it's free.

In [ ]:
buggy = FastAPI()

@buggy.get("/users/{user_id}")          # defined FIRST
def read_user(user_id: str):
    return {"handler": "read_user", "user_id": user_id}

@buggy.get("/users/me")                 # defined SECOND
def read_me():
    return {"handler": "read_me", "user": "the logged-in user"}

bc = TestClient(buggy)
show(bc.get("/users/42"), "GET /users/42")
show(bc.get("/users/me"), "GET /users/me")

**`/users/me` was handled by `read_user`.** FastAPI matches **in definition order, first match
wins** — and `me` is a perfectly good `str`, so the generic route matched and your specific handler
was never consulted.

Note how quiet the failure is: status `200`, plausible JSON, wrong handler. Had `user_id` been typed
`int` you'd have got a `422` instead — same bug, louder symptom, and honestly easier to debug.

### The fix: specific before generic

In [ ]:
fixed = FastAPI()

@fixed.get("/users/me")                 # SPECIFIC first
def read_me_fixed():
    return {"handler": "read_me", "user": "the logged-in user"}

@fixed.get("/users/{user_id}")          # GENERIC second
def read_user_fixed(user_id: str):
    return {"handler": "read_user", "user_id": user_id}

fc = TestClient(fixed)
show(fc.get("/users/42"), "GET /users/42")
show(fc.get("/users/me"), "GET /users/me")

> 🎯 **Remember this** — specific routes go above parameterised ones. Always.

---

## Part 4 — Query parameters

The rule: **a parameter whose name is not in the path is a query parameter.** Required or optional is
decided by the default, exactly as in ordinary Python.

In [ ]:
@app.get("/weather/daily")
def read_daily(
    city: str,                          # no default -> REQUIRED
    limit: int = 10,                    # default    -> optional
    order: str | None = None,           # default None -> optional, and "absent" is detectable
    verbose: bool = False,              # booleans get friendly parsing
):
    return {"city": city, "limit": limit, "order": order, "verbose": verbose}

show(client.get("/weather/daily?city=Utrecht"),                    "city only")
show(client.get("/weather/daily?city=Utrecht&limit=3"),            "with limit")
show(client.get("/weather/daily?city=Utrecht&order=asc"),          "with order")
show(client.get("/weather/daily"),                                 "MISSING city")

### 🔮 Predict

Which of these send `verbose=True`?
`?verbose=true` · `?verbose=True` · `?verbose=1` · `?verbose=yes` · `?verbose=on` · `?verbose=maybe`

In [ ]:
for value in ["true", "True", "1", "yes", "on", "0", "no", "off", "maybe"]:
    r = client.get(f"/weather/daily?city=X&verbose={value}")
    result = r.json().get("verbose") if r.status_code == 200 else f"{r.status_code} rejected"
    print(f"?verbose={value:<8} -> {result}")

### Constraints and documentation with `Query`

In [ ]:
@app.get("/cities")
def list_cities(
    limit: int = Query(default=10, ge=1, le=100, description="Rows to return"),
    name_starts_with: str = Query(default="", max_length=20),
):
    return {"limit": limit, "prefix": name_starts_with}

for qs in ["", "?limit=50", "?limit=0", "?limit=999", "?name_starts_with=" + "x" * 25]:
    show(client.get("/cities" + qs), f"GET /cities{qs[:28]}")

You wrote no `if` statements. `ge=1`, `le=100` and `max_length=20` produced those `422`s, and
they simultaneously became documentation.

### Types you get for free

In [ ]:
from datetime import date
from uuid import UUID

@app.get("/typed")
def typed(when: date, ident: UUID, ratio: float):
    return {"when": str(when), "when_type": type(when).__name__,
            "ident_type": type(ident).__name__, "ratio": ratio}

show(client.get("/typed?when=2026-09-01&ident=123e4567-e89b-12d3-a456-426614174000&ratio=0.5"), "valid")
show(client.get("/typed?when=not-a-date&ident=nope&ratio=abc"), "all three invalid")

One request, three failures, **all reported at once** — the caller fixes everything in a single
round trip. And `when` arrived as a real `datetime.date`, so you can do date arithmetic on it
immediately.

---

## Part 5 — Status codes and errors

In [ ]:
KNOWN = {"Utrecht", "Amsterdam", "Rotterdam"}

@app.post("/cities", status_code=status.HTTP_201_CREATED)
def create_city(name: str):
    return {"created": name}

@app.get("/cities/{name}")
def read_city(name: str):
    if name not in KNOWN:
        raise HTTPException(status_code=404, detail=f"City {name!r} not found")
    return {"city": name}

show(client.post("/cities?name=Zwolle"),   "POST /cities")
show(client.get("/cities/Utrecht"),        "GET /cities/Utrecht")
show(client.get("/cities/Atlantis"),       "GET /cities/Atlantis")

Two details worth pausing on:

- **`raise`, not `return`.** Errors usually happen deep inside helper functions; `return` exits one
  function, `raise` unwinds all of them. Same reason your Module 2 `fetch.py` uses
  `response.raise_for_status()`.
- **`{name!r}`** applies `repr()`, so the value appears in quotes. `City 'Zwolle ' not found`
  immediately shows a trailing space that `City Zwolle  not found` would hide.

---

## Part 6 — The docs wrote themselves

Nobody documented this app. Let's look at what exists anyway.

In [ ]:
schema = client.get("/openapi.json").json()

print("title  :", schema["info"]["title"])
print("paths  :")
for path, methods in schema["paths"].items():
    for method in methods:
        print(f"   {method.upper():<6} {path}")

In [ ]:
# The constraints you declared are IN the schema - which is why /docs can show them.
params = schema["paths"]["/cities"]["get"]["parameters"]
print(json.dumps(params, indent=2)[:700])

`minimum: 1` and `maximum: 100` came from `Query(ge=1, le=100)`. One declaration produced the
conversion, the validation, and the documentation. **Docs derived from code cannot drift out of
date** — that's the answer to the third prediction question from the theory doc.

---

## Part 7 — `APIRouter`: sub-menus

One file is fine now. It isn't at thirty endpoints.

In [ ]:
weather_router = APIRouter(prefix="/weather", tags=["weather"])

@weather_router.get("/weekly")          # final path: /weather/weekly
def read_weekly():
    return {"rows": []}

health_router = APIRouter(tags=["health"])

@health_router.get("/health")
def health():
    return {"status": "ok"}

big = FastAPI()
big.include_router(weather_router)
big.include_router(health_router)

bigc = TestClient(big)
show(bigc.get("/weather/weekly"), "GET /weather/weekly")
show(bigc.get("/health"),         "GET /health")

print()
print("routes assembled from two routers:")
for route in big.routes:
    if hasattr(route, "methods") and not route.path.startswith(("/openapi", "/docs", "/redoc")):
        print("   ", sorted(route.methods), route.path)

`prefix="/weather"` is prepended to every path in that router. **No trailing slash** on the
prefix — `prefix="/weather/"` produces `//weekly`.

This is exactly the structure the project uses: `routers/health.py` and `routers/weather.py`, both
included by `main.py`.

---

## Part 8 — Run the real server

Everything so far used `TestClient`. Now do it for real, from a **terminal** (not this notebook):

```bash
cd module-03-fastapi-azure/lessons/day2-fastapi-basics
```

Create `demo.py`:

```python
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def read_root():
    return {"message": "hello from a real server"}
```

Then:

```bash
uvicorn demo:app --reload
```

| Token | Meaning |
|---|---|
| `uvicorn` | The server programme |
| `demo` | The module — `demo.py` without the extension |
| `:` | Separator |
| `app` | The variable inside that module |
| `--reload` | Restart on save. **Development only** |

Open these three, in this order:

1. <http://127.0.0.1:8000/> — your endpoint
2. <http://127.0.0.1:8000/docs> — **click "Try it out", then "Execute"**. You are sending real requests
3. <http://127.0.0.1:8000/redoc> — the same information, formatted for reading

Stop the server with `Ctrl+C`.

---

## Exercises

### Exercise 1 — A small resource (⭐)

Build an app with:
- `GET /stations` returning a list of three station dicts
- `GET /stations/{code}` returning one, or **404** with a helpful detail if the code is unknown
- `GET /stations/nearest?lat=..&lon=..` — note this must **not** be shadowed by the route above

In [ ]:
# Your code here


<details>
<summary>💡 Solution</summary>

```python
ex1 = FastAPI()

STATIONS = {
    "260": {"code": "260", "name": "De Bilt",   "lat": 52.10, "lon": 5.18},
    "240": {"code": "240", "name": "Schiphol",  "lat": 52.32, "lon": 4.79},
    "344": {"code": "344", "name": "Rotterdam", "lat": 51.95, "lon": 4.44},
}

@ex1.get("/stations")
def list_stations():
    return list(STATIONS.values())

@ex1.get("/stations/nearest")            # SPECIFIC first, or /nearest becomes a code
def nearest(lat: float, lon: float):
    best = min(
        STATIONS.values(),
        key=lambda s: (s["lat"] - lat) ** 2 + (s["lon"] - lon) ** 2,
    )
    return best

@ex1.get("/stations/{code}")             # GENERIC second
def read_station(code: str):
    if code not in STATIONS:
        raise HTTPException(404, detail=f"No station with code {code!r}")
    return STATIONS[code]

c = TestClient(ex1)
print(c.get("/stations").json())
print(c.get("/stations/260").json())
print(c.get("/stations/999").status_code, c.get("/stations/999").json())
print(c.get("/stations/nearest?lat=52.1&lon=5.1").json())
```

The ordering of `nearest` above `{code}` is the whole point of the exercise. Swap them and
`/stations/nearest` returns a 404 for "no station with code 'nearest'" — a confusing message for a
route that exists.
</details>

### Exercise 2 — Constrain and document (⭐⭐)

Write `GET /weather/daily` that:
- **requires** `city` (1–100 characters)
- takes an optional `from_date` as a real `date`
- takes `limit`, between 1 and 500, defaulting to 50
- takes `order`, which must be `"asc"` or `"desc"`, defaulting to `"desc"` — return **422** for anything else
- returns `422` if `from_date` is in the future

In [ ]:
# Your code here


<details>
<summary>💡 Solution</summary>

```python
from datetime import date

ex2 = FastAPI()

@ex2.get("/weather/daily")
def daily(
    city: str = Query(min_length=1, max_length=100),
    from_date: date | None = None,
    limit: int = Query(default=50, ge=1, le=500),
    order: str = Query(default="desc", pattern="^(asc|desc)$"),
):
    if from_date is not None and from_date > date.today():
        raise HTTPException(422, detail="from_date cannot be in the future")
    return {"city": city, "from_date": from_date, "limit": limit, "order": order}

c = TestClient(ex2)
for qs in ["?city=Utrecht", "?city=Utrecht&order=sideways", "?city=Utrecht&limit=9999",
           "?city=", "?city=Utrecht&from_date=2099-01-01"]:
    r = c.get("/weather/daily" + qs)
    print(f"{r.status_code}  {qs}")
```

Two ways to constrain `order`: `pattern="^(asc|desc)$"` as above (declarative, appears in the docs),
or an `Enum` type, which is nicer still because `/docs` renders a dropdown. The future-date check has
to be a manual `if`, because "not in the future" is not a static constraint.
</details>

### Exercise 3 — Read the schema (⭐⭐⭐)

Without opening `/docs` in a browser: using only `client.get("/openapi.json")`, print a table of every
endpoint in your Exercise 2 app showing path, method, and each parameter's name, whether it is
required, and where it comes from (`query` or `path`).

In [ ]:
# Your code here


<details>
<summary>💡 Solution</summary>

```python
schema = TestClient(ex2).get("/openapi.json").json()

for path, methods in schema["paths"].items():
    for method, spec in methods.items():
        print(f"{method.upper()} {path}")
        for p in spec.get("parameters", []):
            required = "required" if p.get("required") else "optional"
            print(f"    {p['name']:<12} {p['in']:<6} {required:<9} {p['schema']}")
```

This is exactly how client generators, API gateways, and test tools consume your service. Because
the schema is derived from your signatures rather than maintained by hand, anything that reads it is
always looking at the truth.
</details>

---

## ✅ Before you move on

Without scrolling:

- What does `@app.get("/x")` do, and *when* does it do it?
- Where does a parameter come from if its name is not in the path?
- What makes a query parameter optional?
- Why must `/users/me` be defined above `/users/{user_id}`?
- What are the three tokens in `uvicorn main:app`?

Next: **[Day 3 theory](../day3-pydantic-validation/05_theory_pydantic_models.md)** — the loose dicts
you've been returning get a shape, and the doorman gets much stricter.